# R18 Kinetics Processing (Publication Notebook)

This notebook processes gated FlowJo CSV exports that were already split by `FCS_splitter.py`.

## Workflow
1. Read FL2 loading traces and compute per-trace normalization factors from `AUCSTD`.
2. Read FL1 `initial`, `loading`, `baseline`, and `activated` phases.
3. Apply AUC-based normalization to FL1 and assemble the full normalized FL1 curve.
4. Report per-trace resting (baseline) and activated FL1 levels.
5. Export per-folder assembled curves and a master summary table.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Gold-standard FL2 AUC from `Normalization_estimator.py`
AUCSTD = 1259951.4009151116

# Activated-value extraction window from activated segment.
# Start at +10 points to skip edge effects; ignore the final 50 points.
ACTIVATED_LEFT_TRIM = 10
ACTIVATED_RIGHT_TRIM = 50


In [ ]:
def _read_phase(folder: Path, phase: str, channel: str) -> pd.DataFrame:
    """Read one split phase table produced by `FCS_splitter.py`."""
    return pd.read_csv(folder / f"Loading_Baseline/{phase}_{channel}.csv")


def _safe_activated_window(series: pd.Series,
                           left_trim: int = ACTIVATED_LEFT_TRIM,
                           right_trim: int = ACTIVATED_RIGHT_TRIM) -> pd.Series:
    """Return trimmed activated segment; fall back to full series if too short."""
    clean = series.dropna()
    if len(clean) <= left_trim + right_trim:
        return clean
    return clean.iloc[left_trim: len(clean) - right_trim]


def get_normalization(folder: Path, auc_std: float = AUCSTD) -> pd.Series:
    """Compute per-trace normalization coefficient from FL2 loading AUC.

    norm_coeff = AUC(loading_fl2_trace) / auc_std
    """
    fl2_loading = _read_phase(folder, "loading", "fl2")
    norm_values = {}

    for col in fl2_loading.columns:
        auc_fl2 = float(np.trapz(fl2_loading[col].dropna()))
        norm_values[col] = auc_fl2 / auc_std

    return pd.Series(norm_values, name="norm_coeff")


def assemble_fl1(folder: Path, normalize: pd.Series):
    """Assemble normalized FL1 curve and extract resting/activated values.

    Returns:
        tuple[pd.Series, pd.Series, pd.DataFrame]:
            resting_values (%), activated_values (%), assembled_curve
    """
    initial = _read_phase(folder, "initial", "fl1")
    loading = _read_phase(folder, "loading", "fl1")
    baseline = _read_phase(folder, "baseline", "fl1")
    activated = _read_phase(folder, "activated", "fl1")

    initial_means = initial.mean(axis=0, skipna=True)

    assembled = {}
    resting_values = {}
    activated_values = {}

    common_cols = [
        col for col in loading.columns
        if col in baseline.columns and col in activated.columns and col in initial.columns and col in normalize.index
    ]

    for col in common_cols:
        # AUC-normalized post-initial trace: (loading + baseline + activated)
        post_initial = pd.concat([loading[col], baseline[col], activated[col]], ignore_index=True)
        post_initial = post_initial.multiply(normalize[col]).divide(initial_means[col])

        # Full curve includes initial segment normalized only to initial mean.
        assembled[col] = pd.concat(
            [initial[col].divide(initial_means[col]), post_initial],
            ignore_index=True,
        )

        resting_values[col] = float(np.nanmedian(baseline[col] / initial_means[col] * normalize[col]) * 100)

        act_window = _safe_activated_window(activated[col])
        activated_values[col] = float(np.nanmedian(act_window / initial_means[col] * normalize[col]) * 100)

    assembled_df = pd.DataFrame(assembled)
    assembled_df.to_excel(folder / "Analyzed.xlsx")

    plt.figure(figsize=(10, 5))
    for col in assembled_df.columns:
        plt.plot(assembled_df[col], label=col, alpha=0.7)
    plt.title(f"Assembled normalized FL1 traces: {folder.name}")
    plt.xlabel("Time bins")
    plt.ylabel("Normalized FL1")
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.show()

    return pd.Series(resting_values), pd.Series(activated_values), assembled_df


## Run Batch Processing

Set `master_folder` to the directory containing experiment subfolders.
Each subfolder is expected to include `Loading_Baseline/*.csv` outputs from `FCS_splitter.py`.


In [ ]:
master_folder = Path('/Users/alexeymartyanov/Library/CloudStorage/OneDrive-UniversityofNorthCarolinaatChapelHill/Projects/EC Integrins/FACS/Kindlins/KindlinDataset')
sub_folders = sorted(p for p in master_folder.iterdir() if p.is_dir())

base_list = []
act_list = []

for sub_folder in sub_folders:
    norm_coeff = get_normalization(sub_folder, auc_std=AUCSTD)
    baselines, activated_vals, _assembled = assemble_fl1(sub_folder, norm_coeff)

    # Preserve original behavior: append per-folder series; duplicates by sample name are allowed.
    base_list.append(baselines)
    act_list.append(activated_vals)

base_fin = pd.concat(base_list, axis=0)
act_fin = pd.concat(act_list, axis=0)

resultant = pd.DataFrame({
    'Resting': base_fin,
    'Activated': act_fin,
})

resultant.to_excel(master_folder / 'result.xlsx')
resultant.head()
